In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import time

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument('--headless')
# options.add_argument('--disable-gpu')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920x1080')
options.add_argument('--start-maximized')
# options.add_argument('--user-Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36')
options.add_argument('--lang=ko_KR')

driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.musinsa.com/main/musinsa/recommend?gf=A")
time.sleep(3)

search_button = driver.find_element(By.CSS_SELECTOR, ".sc-dzagxs-2")
search_button.click()
time.sleep(2)
popular_keywords = driver.find_elements(By.CSS_SELECTOR, ".search-home-popular-wrap .sc-1xzm8bl-1")
target_keyword = popular_keywords[1].text
print("목표키워드 : ", target_keyword)

search_input = driver.find_element(By.CLASS_NAME, "search-home-search-bar-keyword")
search_input.clear()
search_input.send_keys(target_keyword)
time.sleep(1)

search_input.send_keys(Keys.RETURN)
time.sleep(5)

print("탐색완료 : ", driver.title)
driver.quit()

목표키워드 :  반팔
탐색완료 :  반팔 | 무신사 추천 상품


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re 

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument('--headless')
# options.add_argument('--disable-gpu')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920x1080')
options.add_argument('--start-maximized')
#options.add_argument('--user-Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36')
options.add_argument('--lang=ko_KR')

driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.musinsa.com/main/musinsa/recommend?gf=A")
time.sleep(3)

search_button = driver.find_element(By.CSS_SELECTOR, ".sc-dzagxs-2")
search_button.click()
time.sleep(2)
popular_keywords = driver.find_elements(By.CSS_SELECTOR, ".search-home-popular-wrap .sc-1xzm8bl-1")
target_keyword = popular_keywords[1].text
print("목표키워드 : ", target_keyword)

search_input = driver.find_element(By.CLASS_NAME, "search-home-search-bar-keyword")
search_input.clear()
search_input.send_keys(target_keyword)
time.sleep(1)

search_input.send_keys(Keys.RETURN)
time.sleep(5)

print("탐색완료 : ", driver.title)

scroll_pause = 2

for _ in range(10) :
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(scroll_pause)

products = driver.find_elements(By.CSS_SELECTOR, "div.sc-d36st-1.elXjMR")[:100]
product_data = []

for product in products :
    try :
        name = product.find_element(By.CSS_SELECTOR, "div.sc-cMuefe.jIlLeV a.gtm-select-item span").text.strip()
        # print(name)
        url = product.find_element(By.CSS_SELECTOR, "div.sc-cMuefe.jIlLeV a.gtm-select-item").get_attribute("href")
        # print(url)
        review_info = product.find_elements(By.CSS_SELECTOR, "span.text-etc_11px_reg.text-yellow.font-pretendard")[1].text.strip()
        # print(review_info)
        review_elements = product.find_elements(By.CSS_SELECTOR, "span.text-etc_11px_reg.text-yellow.font-pretendard")
        review_info = review_elements[1].text if len(review_elements) > 1 else ""
        review_count = int(re.sub(r"[^\d]", "", review_info)) if review_info else 0
        product_data.append({"상품명": name, "링크": url, "리뷰수": review_count})
        
        
    except :
        continue

df = pd.DataFrame(product_data)
df_top30 = df.sort_values(by="리뷰수", ascending=False).head(30).reset_index(drop=True)
            
driver.quit()
print(df_top30.head())

목표키워드 :  반팔
탐색완료 :  반팔 | 무신사 추천 상품
                                  상품명  \
0      PLUR SS TEE WHITE(CV2BMMT515A)   
1                    오리엔탈 워크샵 반팔티 화이트   
2                        세일링 요트클럽 티셔츠   
3  XTT091 미누의 트라이벌 스포티 반팔 티셔츠 (BLACK)   
4              CANDY STRIPE T / BLACK   

                                         링크  리뷰수  
0  https://www.musinsa.com/products/1849317  934  
1  https://www.musinsa.com/products/3182406  865  
2  https://www.musinsa.com/products/4065364  817  
3  https://www.musinsa.com/products/3890825  751  
4  https://www.musinsa.com/products/4142537  740  


In [3]:
from openpyxl import Workbook

driver = webdriver.Chrome(service=service, options=options)
all_reviews = []
for idx, row in df_top30.iterrows() :
    print(f"{idx+1}번째 상품리뷰 수집 중 :", {row['상품명']})
    driver.get(row['링크'])
    time.sleep(2)

    try :
        driver.execute_script("window.scrollBy(0, 200);")
        time.sleep(1)
        review_button = driver.find_element(By.XPATH, "//button[@data-button-id='prd_review_tab']")
        review_button.click()
        time.sleep(2)
        for _ in range(2) :
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
            time.sleep(1.5)
        review_elements = driver.find_elements(By.CSS_SELECTOR,".text-body_13px_reg.text-black.font-pretendard")
        for review in review_elements[:10] :
            text = review.text.strip()
            if text :
                all_reviews.append({
                    "상품명" : row["상품명"],
                    "리뷰" : text
                })
    except Exception as e :
        print("리뷰 수집 실패 : ", e)
        continue
driver.quit()

wb = Workbook()
ws = wb.active
ws.title = "무신사리뷰"

ws.append(["상품명", "리뷰"])

for r in all_reviews :
    ws.append([r["상품명"], r["리뷰"]])

wb.save("musinsa_top30_reviews.xlsx")
print("엑셀 저장 완료")


1번째 상품리뷰 수집 중 : {'PLUR SS TEE WHITE(CV2BMMT515A)'}
2번째 상품리뷰 수집 중 : {'오리엔탈 워크샵 반팔티 화이트'}
3번째 상품리뷰 수집 중 : {'세일링 요트클럽 티셔츠'}
4번째 상품리뷰 수집 중 : {'XTT091 미누의 트라이벌 스포티 반팔 티셔츠 (BLACK)'}
5번째 상품리뷰 수집 중 : {'CANDY STRIPE T / BLACK'}
6번째 상품리뷰 수집 중 : {'썬 레트로 그래픽 반팔 티셔츠 레트로 차콜'}
7번째 상품리뷰 수집 중 : {'노미네이츠 피그먼트 반팔티셔츠 딥그레이 SHST6789'}
8번째 상품리뷰 수집 중 : {'ship t-shirt(white)'}
9번째 상품리뷰 수집 중 : {'[한소희 착용] 미니핏 그래픽 티셔츠 395504 [21color]'}
10번째 상품리뷰 수집 중 : {'C 90S LINE BASIC T-SHIRT_NAVY'}
11번째 상품리뷰 수집 중 : {'ML GARMENTS CREW [PIGMENT D.GREY]'}
12번째 상품리뷰 수집 중 : {'로고 배색 져지 [블랙]'}
13번째 상품리뷰 수집 중 : {'[2PACK] 어센틱 로고 반팔티셔츠'}
14번째 상품리뷰 수집 중 : {'DOODLE HEART RINGER HALF T WHITE'}
15번째 상품리뷰 수집 중 : {'슬림 뱃지 립 티셔츠 (T32E1TTO06TWT1YBR)'}
16번째 상품리뷰 수집 중 : {'트래블 티셔츠 블랙'}
17번째 상품리뷰 수집 중 : {'VSW Cartoon 1 T-Shirts White'}
18번째 상품리뷰 수집 중 : {'[2PACK] 아치로고 오버사이즈 반팔 티셔츠_4컬러'}
19번째 상품리뷰 수집 중 : {'Digital Tattoo Tee Renewal Mint'}
20번째 상품리뷰 수집 중 : {'아치로고 오버사이즈 반팔 티셔츠 [차콜]'}
21번째 상품리뷰 수집 중 : {'set 89 t (dark grey)'}
22번째 상품리뷰 수집 중 : {'DOUBLE 

In [28]:
pip install sqlalchemy

In [37]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pymysql
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus


EXCEL_PATH = "musinsa_top30_reviews.xlsx"
DB_NAME = "musinsa_db_v4"
TABLE_NAME = "reviews"
USER = "root"
PASSWORD = "@!Dydwls900"
ENCODED_PASSWORD = quote_plus(PASSWORD)
HOST = "127.0.0.1"
PORT = 3306

df = pd.read_excel(EXCEL_PATH)
conn = pymysql.connect(host=HOST, user=USER, password=PASSWORD, port=PORT)
cursor = conn.cursor()
cursor.execute(
    f"CREATE DATABASE IF NOT EXISTS {DB_NAME} CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"
)
conn.close()

engine_str = f"mysql+pymysql://{USER}:{ENCODED_PASSWORD}@{HOST}:{PORT}/{DB_NAME}?charset=utf8mb4"
engine = create_engine(engine_str)

df.to_sql(name=TABLE_NAME, con=engine, if_exists="replace", index=False)
print(f"데이터가 MySQL {DB_NAME}의 테이블 {TABLE_NAME}에 성공적으로 저장되었습니다.")

데이터가 MySQL musinsa_db_v4의 테이블 reviews에 성공적으로 저장되었습니다.


In [5]:
pip install gspread # 파이썬에서 구글 스프레드시터에 접근 가능


   ------------- -------------------------- 2/6 [requests-oauthlib]
   --------------------------------- ------ 5/6 [gspread]
   ---------------------------------------- 6/6 [gspread]

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install oauth2client # 구글 접속 허가


   ---------------------------------------- 2/2 [oauth2client]

Note: you may need to restart the kernel to use updated packages.


In [18]:
from konlpy.tag import Okt
from oauth2client.service_account import ServiceAccountCredentials
import gspread
import pandas as pd

df = pd.read_excel("musinsa_top30_reviews.xlsx")

okt = Okt()
df["리뷰_키워드"] = df["리뷰"].apply(lambda x: ", ".join(okt.nouns(str(x))))

scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("airy-cortex-468703-a0-155e0b07221d.json", scope)
client = gspread.authorize(creds)

spreadsheet = client.open("무신사_리뷰_분석_v.01")
sheet1 = spreadsheet.sheet1
sheet1.clear()
sheet1.update([df.columns.values.tolist()] + df.values.tolist())

try :
    spreadsheet.add_worksheet(title="키워드 분석", rows="1000", cols="20")
except :
    pass

sheet2 = spreadsheet.worksheet("키워드 분석")
sheet2.clear()
sheet2.update([["상품명", "리뷰", "리뷰_키워드"]] + df[["상품명", "리뷰", "리뷰_키워드"]].values.tolist())

print("구글 스프레듯트에 원본 + 키워드 분석자료 업데이트 완료")

구글 스프레듯트에 원본 + 키워드 분석자료 업데이트 완료
